# Customer Upsell Propensity Training

End-to-end notebook to generate clean customer data, train and evaluate upsell propensity models, and export artifacts for serving and downstream checks.


In [ ]:
import sys
print(sys.executable)

import pandas as pd
print("pandas version:", pd.__version__)

/Users/muhammadabdullah/offer-ranker-api/.venv/bin/python
pandas version: 2.3.3


## Setup


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

RANDOM_SEED = 42
sns.set(style="whitegrid")

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Models directory: {MODELS_DIR.resolve()}")


Data directory: /Users/muhammadabdullah/offer-ranker-api/training/data
Models directory: /Users/muhammadabdullah/offer-ranker-api/training/models


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

RANDOM_SEED = 42
sns.set(style="whitegrid")

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Models directory: {MODELS_DIR.resolve()}")


Data directory: /Users/muhammadabdullah/offer-ranker-api/training/data
Models directory: /Users/muhammadabdullah/offer-ranker-api/training/models


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

RANDOM_SEED = 42
sns.set(style="whitegrid")

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Models directory: {MODELS_DIR.resolve()}")


Data directory: /Users/muhammadabdullah/offer-ranker-api/training/data
Models directory: /Users/muhammadabdullah/offer-ranker-api/training/models


## Data generation and loading


In [ ]:
csv_path = DATA_DIR / "customers.csv"

if not csv_path.exists():
    rng = np.random.default_rng(RANDOM_SEED)
    n_samples = 2000

    ages = rng.integers(18, 80, size=n_samples)
    tenure_months = rng.integers(0, 120, size=n_samples)
    monthly_spend = rng.uniform(20.0, 500.0, size=n_samples)
    num_support_tickets = rng.integers(0, 12, size=n_samples)

    logits = (
        0.03 * ages
        + 0.04 * tenure_months
        + 0.06 * monthly_spend
        - 0.25 * num_support_tickets
        - 25
    )
    probs = 1 / (1 + np.exp(-logits))
    is_upsell_accepted = rng.binomial(1, probs)

    df_generated = pd.DataFrame(
        {
            "customer_id": np.arange(1, n_samples + 1, dtype=int),
            "age": ages,
            "tenure_months": tenure_months,
            "monthly_spend": monthly_spend,
            "num_support_tickets": num_support_tickets,
            "is_upsell_accepted": is_upsell_accepted,
        }
    )

    df_generated.to_csv(csv_path, index=False)
    print(f"Generated synthetic dataset and saved to {csv_path}")
else:
    print(f"Found existing dataset at {csv_path}")

# Always load the dataset
df = pd.read_csv(csv_path)
print(df.shape)
df.head()


Found existing dataset at data/customers.csv
(2000, 6)


,customer_id,age,tenure_months,monthly_spend,num_support_tickets,is_upsell_accepted
0,1,23,105,424.511345,7,1
1,2,65,7,234.266956,8,0
2,3,58,47,477.428078,3,1
3,4,45,54,332.381403,4,0
4,5,44,95,75.631455,11,0


## Exploratory data analysis


In [ ]:
display(df.head())
print("\nInfo:")
df_info = df.info()
print("\nDescribe:")
display(df.describe())
print("\nNulls per column:")
display(df.isnull().sum())


,customer_id,age,tenure_months,monthly_spend,num_support_tickets,is_upsell_accepted
0,1,23,105,424.511345,7,1
1,2,65,7,234.266956,8,0
2,3,58,47,477.428078,3,1
3,4,45,54,332.381403,4,0
4,5,44,95,75.631455,11,0



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          2000 non-null   int64  
 1   age                  2000 non-null   int64  
 2   tenure_months        2000 non-null   int64  
 3   monthly_spend        2000 non-null   float64
 4   num_support_tickets  2000 non-null   int64  
 5   is_upsell_accepted   2000 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 93.9 KB

Describe:


,customer_id,age,tenure_months,monthly_spend,num_support_tickets,is_upsell_accepted
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.00000
mean,1000.500000,48.490500,59.779000,256.746959,5.510500,0.25950
std,577.494589,17.986263,34.493246,139.128810,3.390669,0.43847
min,1.000000,18.000000,0.000000,20.308306,0.000000,0.00000
25%,500.750000,33.000000,29.000000,137.995918,3.000000,0.00000
50%,1000.500000,49.000000,59.000000,257.621976,6.000000,0.00000
75%,1500.250000,64.000000,90.000000,377.914277,8.000000,1.00000
max,2000.000000,79.000000,119.000000,499.887362,11.000000,1.00000



Nulls per column:


customer_id            0
age                    0
tenure_months          0
monthly_spend          0
num_support_tickets    0
is_upsell_accepted     0
dtype: int64

## Preprocessing and train/test split


In [ ]:
feature_cols = ["age", "tenure_months", "monthly_spend", "num_support_tickets"]
target_col = "is_upsell_accepted"

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[("num", numeric_transformer, feature_cols)]
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


Train shape: (1600, 4), Test shape: (400, 4)


## Modeling with Logistic Regression and Random Forest


In [ ]:
models = {
    "log_reg": {
        "estimator": LogisticRegression(max_iter=500, random_state=RANDOM_SEED),
        "param_grid": {
            "model__C": [0.1, 1.0, 10.0],
            "model__penalty": ["l2"],
            "model__solver": ["lbfgs"],
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(random_state=RANDOM_SEED),
        "param_grid": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 5],
        },
    },
}

results = []
best_estimators = {}

for name, cfg in models.items():
    print(f"Training model: {name}")
    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", cfg["estimator"]),
        ]
    )

    search = GridSearchCV(
        pipeline,
        cfg["param_grid"],
        cv=3,
        n_jobs=-1,
        scoring="roc_auc",
    )

    search.fit(X_train, y_train)
    best_pipe = search.best_estimator_
    best_estimators[name] = best_pipe


    y_pred = best_pipe.predict(X_test)
    y_proba = best_pipe.predict_proba(X_test)[:, 1]

    metrics = {
        "model": name,
        "best_params": search.best_params_,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    results.append(metrics)

results_df = pd.DataFrame(results).set_index("model")
results_df



SyntaxError: unterminated string literal (detected at line 24) (2042871848.py, line 24)

## Champion selection


In [ ]:
champion_name = results_df["roc_auc"].idxmax()
champion_model = best_estimators[champion_name]
champion_metrics = results_df.loc[champion_name]

print("Champion model:", champion_name)
print("Metrics:'\'", champion_metrics)


The champion model is chosen based on the highest ROC AUC on the held-out test set, balancing true positive and false positive rates. A higher ROC AUC indicates better discrimination between customers likely to accept the upsell and those who are not.


## Visual evaluation: ROC curves and confusion matrix


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, estimator in best_estimators.items():
    RocCurveDisplay.from_estimator(
        estimator,
        X_test,
        y_test,
        name=name,
        ax=ax,
    )
ax.plot([0, 1], [0, 1], "k--", label="chance")
ax.set_title("ROC Curves")
ax.legend()
plt.show()

ConfusionMatrixDisplay.from_estimator(
    champion_model,
    X_test,
    y_test,
    cmap="Blues",
)
plt.title(f"Confusion Matrix: {champion_name}")
plt.show()


## Feature importance / coefficients for the champion


In [ ]:
model_step = champion_model.named_steps["model"]
feature_importance_df = None

if hasattr(model_step, "feature_importances_"):
    importance = model_step.feature_importances_
    feature_importance_df = pd.DataFrame(
        {"feature": feature_cols, "importance": importance}
    ).sort_values("importance", ascending=False)
elif hasattr(model_step, "coef_"):
    coefs = model_step.coef_[0]
    feature_importance_df = pd.DataFrame(
        {"feature": feature_cols, "importance": coefs}
    ).sort_values("importance", key=abs, ascending=False)

if feature_importance_df is not None:
    display(feature_importance_df)
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=feature_importance_df,
        x="importance",
        y="feature",
        orient="h",
        palette="Blues_d",
    )
    plt.title(f"Feature influence: {champion_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Champion model does not expose feature importance or coefficients.")


The most influential features above indicate how customer characteristics drive upsell acceptance. Higher positive coefficients or importances mean that increases in those features make upsell acceptance more likely, while negative values imply the opposite.


## Export artifacts for serving


In [ ]:
model_path = MODELS_DIR / "offer_model.joblib"
schema_path = MODELS_DIR / "input_schema.json"

joblib.dump(champion_model, model_path)
print(f"Saved champion pipeline to {model_path}")

input_schema = {
    "type": "object",
    "properties": {
        "age": {"type": "number"},
        "tenure_months": {"type": "number"},
        "monthly_spend": {"type": "number"},
        "num_support_tickets": {"type": "number"},
    },
    "required": [
        "age",
        "tenure_months",
        "monthly_spend",
        "num_support_tickets",
    ],
}

schema_path.write_text(json.dumps(input_schema, indent=2))
print(f"Wrote input schema to {schema_path}")


## How this fits into MLOps


- Project 3 (dq-check) can validate `data/customers.csv` before any training to ensure schema and ranges stay clean.
- This notebook generates the dataset when missing, explores it, trains/evaluates models, and selects a champion.
- The OfferRanker API (Project 1) loads `models/offer_model.joblib` and `models/input_schema.json` produced here to serve predictions consistently.


## How to run this notebook


1. `python3 -m venv .venv && source .venv/bin/activate`
2. `pip install pandas numpy scikit-learn matplotlib seaborn joblib`
3. Open `training/customer_upsell_propensity.ipynb` in Jupyter/VS Code.
4. Run all cells from top to bottom to generate data, train models, and export artifacts.
